# Task 1: Data Cleaning — Build `trends_combined_english.csv`

Loads the 5 trend-filtered CSVs, recomputes `is_covid_framed` with a corrected keyword list,
combines them, filters to English-only, and drops rows unusable for the outcome variables.

Out of scope: USDA/grocery/trade data (untouched, stays in `code/02_data_cleaning.ipynb`).

In [1]:
import pandas as pd
import numpy as np

## Step 1: Load each of the 5 CSVs, add a `trend` column, recompute `is_covid_framed`

In [2]:
trend_files = {
    "feta_pasta": "feta_pasta.csv",
    "sourdough": "sourdough.csv",
    "banana_bread": "banana_bread.csv",
    "baked_oats": "baked_oats.csv",
    "dalgona_coffee": "dalgona_coffee.csv",
}

covid_keywords = ["covid", "coronavirus", "pandemic", "quarantine", "lockdown"]
covid_pattern = "|".join(covid_keywords)

dfs = []
for trend_name, fname in trend_files.items():
    d = pd.read_csv(f"../data/{fname}")
    d["trend"] = trend_name
    d["is_covid_framed"] = (
        d["text"].str.contains(covid_pattern, case=False, na=False, regex=True) |
        d["hashtags"].str.contains(covid_pattern, case=False, na=False, regex=True)
    )
    dfs.append(d)

combined = pd.concat(dfs, ignore_index=True)
print(combined.shape)
print(combined["trend"].value_counts())

(190704, 23)
trend
sourdough         136692
banana_bread       27153
dalgona_coffee     26402
feta_pasta           332
baked_oats           125
Name: count, dtype: int64


Confirm against known per-trend totals: feta_pasta=332, sourdough=136,692, banana_bread=27,153,
baked_oats=125, dalgona_coffee=26,402; total=190,704.

## Step 2: Filter to English-only

In [3]:
combined = combined[combined["lang"] == "en"].copy()
print(combined.shape)

(126730, 23)


Confirm `combined.shape` matches the known English total: 126,730 rows.

## Step 3: Drop rows with null `statistics.like_count` or `statistics.comment_count`

In [4]:
before = len(combined)
combined = combined.dropna(subset=["statistics.like_count", "statistics.comment_count"]).copy()
print(f"Dropped {before - len(combined)} rows with null likes/comments")
print(combined.shape)

Dropped 320 rows with null likes/comments
(126410, 23)


## Step 4: Do NOT filter on null `hashtags`

A null/missing `hashtags` value is legitimate (a post can have no hashtags and still be usable),
not something to exclude. When `hashtag_count` is built in Task 3, null should become `0`, not an
excluded row. Just confirming the null rate here for documentation.

In [5]:
null_hashtags_pct = combined["hashtags"].isna().mean() * 100
print(f"Null hashtags: {null_hashtags_pct:.1f}% of rows (expected, not a bug, no filtering applied)")

Null hashtags: 10.7% of rows (expected, not a bug, no filtering applied)


## Step 5: Confirm `text` has no nulls (sanity check, not a filtering step)

In [6]:
null_text_count = combined["text"].isna().sum()
print(f"Null text count (should be 0): {null_text_count}")
assert null_text_count == 0, "Unexpected: null text found after English filter, investigate before proceeding"

Null text count (should be 0): 0


## Step 6: Save the result

In [7]:
combined.to_csv("../output/cleaned_data/trends_combined_english.csv", index=False)
print(f"Saved trends_combined_english.csv, shape: {combined.shape}")

Saved trends_combined_english.csv, shape: (126410, 23)


## Final checks against expected results

- Final shape should be **(126,410, 23)**
- Per-trend `is_covid_framed` counts after correction and English filtering:
  feta_pasta=19, sourdough=7,769, banana_bread=3,583, baked_oats=46, dalgona_coffee=2,840
- `lang` column entirely `"en"` at this point

In [8]:
print("Final shape:", combined.shape)
print()
print("is_covid_framed counts by trend:")
print(combined.groupby("trend")["is_covid_framed"].sum())
print()
print("lang value counts (should be entirely 'en'):")
print(combined["lang"].value_counts())
print()
print("Null checks: text=%d, likes=%d, comments=%d" % (
    combined["text"].isna().sum(),
    combined["statistics.like_count"].isna().sum(),
    combined["statistics.comment_count"].isna().sum(),
))

Final shape: (126410, 23)

is_covid_framed counts by trend:
trend
baked_oats          46
banana_bread      3583
dalgona_coffee    2840
feta_pasta          19
sourdough         7769
Name: is_covid_framed, dtype: int64

lang value counts (should be entirely 'en'):
lang
en    126410
Name: count, dtype: int64

Null checks: text=0, likes=0, comments=0
